In [14]:
import json
from pathlib import Path

In [15]:
rules_path = Path("../data/rules/contract_rules.json")

In [16]:
with open(rules_path,'r',encoding='utf-8') as f:
    rules = json.load(f)
print(type(relus))
print(len(rules))
print(rules[0]['title'])

NameError: name 'relus' is not defined

In [17]:
contract_text = """
6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，
乙方有权按照原污水处理单价的1.5倍收取超标处理费用。
"""

In [18]:
rule = rules[0]

prompt = f"""
你是一名工业污水处理合同审查助手。

请严格依据以下审查规则，检查提供的合同条款。

审查规则：
{rule['review_question']}

审查要求：
{rule['expected_requirement']}

合同原文：
{contract_text}

请回答：
1. 该条款是否满足审查要求？
2. 找出存在问题的原文。
3. 解释具体风险。
4. 给出修改建议。

注意：
只根据提供的合同片段审查。
没有出现的条款不能认定整份合同一定缺失。
不得编造合同内容。
"""

print(prompt)


你是一名工业污水处理合同审查助手。

请严格依据以下审查规则，检查提供的合同条款。

审查规则：
合同是否明确水质超标时的处理方式、收费机制，以及严重超标情况下的价格协商机制？

审查要求：
合同应明确超标污水的拒收或接收条件、超标处理费用，以及现有计价规则不适用时的协商处理机制。

合同原文：

6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，
乙方有权按照原污水处理单价的1.5倍收取超标处理费用。


请回答：
1. 该条款是否满足审查要求？
2. 找出存在问题的原文。
3. 解释具体风险。
4. 给出修改建议。

注意：
只根据提供的合同片段审查。
没有出现的条款不能认定整份合同一定缺失。
不得编造合同内容。



In [19]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("SILICONFLOW_API_KEY")
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V4.1-Flash",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key)

In [20]:
response = llm.invoke(prompt)
print(response.content)

1. **该条款是否满足审查要求？**  
   仅就该条款而言，**不满足/未完整满足审查要求**。该条款明确了超标处理费用为原单价1.5倍，部分满足“收费机制”；但**未明确超标污水的拒收或接收条件**，也**未明确严重超标或现有计价规则不适用时的价格协商机制**。不能据此认定整份合同其他条款一定缺失，但该条款本身不完整。

2. **存在问题的原文。**  
   > “甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。”

   问题点：仅约定超过接收标准即按1.5倍收费，未区分超标程度，未约定乙方是否可拒收及拒收/接收条件，也未约定严重超标时价格如何协商。

3. **具体风险。**  
   - **拒收/接收条件不明确**：污水超标时，乙方是否必须接收、是否有权拒收、在何种情形下拒收，条款未明确，易产生操作争议。
   - **收费机制过于单一**：只要超标即按1.5倍收费，未区分轻微超标与严重超标；严重超标导致处理成本大幅增加时，1.5倍可能不足以覆盖成本或引发价格争议。
   - **严重超标协商机制缺失**：当超标严重或现有1.5倍计价规则不适用时，缺少价格协商、补充定价及协商不成的处理路径，可能导致费用无法确定。
   - **“有权收取”表述不完整**：仅赋予乙方收费权，但未同步明确接收/拒收权及相应条件。

4. **修改建议。**  
   建议补充明确以下内容，可修改为类似条款：

   > **6.3 水质超标处理费**  
   > 6.3.1 甲方交付的污水水质超过本协议约定接收标准时，乙方有权根据超标项目、超标程度及对污水处理系统的影响，决定接收或拒收；具体拒收/接收条件为：【双方约定】。  
   > 6.3.2 乙方接收超标污水的，超标处理费用按原污水处理单价的1.5倍收取，适用条件为：【明确适用范围】。  
   > 6.3.3 发生严重超标【可约定为超过接收标准一定倍数、多项指标同时超标或导致处理系统无法正常运行等具体情形】或现有计价规则不适用时，双方应在【约定期限】内另行协商处理价格，并签订补充协议；协商不成的，按本合同争议解决条款处理。

   上述修改仅针对该条款本身；片段未体现的其他内容，不据此认定整份合同必然缺失。


In [21]:
from pydantic import BaseModel,Field
from typing import Literal

class ConstractReviewResult(BaseModel):
    rule_id:str = Field(
        description='审查规则编号'
    )
    status:Literal['符合要求','存在问题','无法判断'] = Field(
        description='依据当前提供的合同文本作出的审查判断'
    )
    evidence: str = Field(
        description="支持审查结论的合同原文，不得编造"
    )

    issue: str = Field(
        description="发现的问题及其具体原因"
    )

    suggestion: str = Field(
        description="针对问题提出的合同修改建议"
    )

In [23]:
print(ConstractReviewResult.model_json_schema())

{'properties': {'rule_id': {'description': '审查规则编号', 'title': 'Rule Id', 'type': 'string'}, 'status': {'description': '依据当前提供的合同文本作出的审查判断', 'enum': ['符合要求', '存在问题', '无法判断'], 'title': 'Status', 'type': 'string'}, 'evidence': {'description': '支持审查结论的合同原文，不得编造', 'title': 'Evidence', 'type': 'string'}, 'issue': {'description': '发现的问题及其具体原因', 'title': 'Issue', 'type': 'string'}, 'suggestion': {'description': '针对问题提出的合同修改建议', 'title': 'Suggestion', 'type': 'string'}}, 'required': ['rule_id', 'status', 'evidence', 'issue', 'suggestion'], 'title': 'ConstractReviewResult', 'type': 'object'}


In [24]:
structured_llm = llm.with_structured_output(ConstractReviewResult)

In [26]:
response = structured_llm.invoke(prompt)

ValidationError: 1 validation error for ConstractReviewResult
  Invalid JSON: invalid number at line 1 column 3 [type=json_invalid, input_value='1. **是否满足审查...协商处理机制**。', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

In [27]:
review_prompt = prompt + f"\n审查规则编号：{rule['rule_id']}。请按照指定的结构化字段输出。"

In [28]:
result = structured_llm.invoke(review_prompt)

print(type(result))
print(result.model_dump())

<class '__main__.ConstractReviewResult'>
{'rule_id': 'WQ-001', 'status': '存在问题', 'evidence': '6.3 水质超标处理费：甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。', 'issue': '该条款仅约定水质超标时按原污水处理单价1.5倍收费，未明确超标污水的拒收或接收条件，未区分严重超标情形，也未约定现有计价规则不适用时的价格协商机制，不满足审查要求。', 'suggestion': '建议补充约定：1）明确水质超标时乙方的拒收或接收条件，例如按超标项目、超标倍数、持续时长等划分可接收与拒收情形；2）明确超标处理费的适用条件和计算方式，并说明是否仅适用于轻微或一般超标；3）增加严重超标情形下的处理机制，包括停止接收、限期整改、另行协商处理价格或委托第三方处理等；4）明确协商不成的后续处理路径。'}


In [29]:
good_contract = """
6.3 水质超标处理费

甲方交付的污水超过接收标准，但超标倍数不超过2倍，
且经乙方评估确认可以处理的，
经乙方书面同意接收后，按照原污水处理单价的1.5倍收费。

当任一指标超标超过2倍，或可能影响污水处理系统安全稳定运行时，
乙方有权拒绝接收。

如双方拟继续接收该类严重超标污水，
须在接收前另行书面协商处理价格。
未能达成一致的，乙方有权拒收。
"""

In [30]:
good_prompt = review_prompt.replace(
    contract_text,
    good_contract
)

In [35]:
good_prompt += """
请严格按照以下要求生成结构化结果：

1. status 只能是：符合要求、存在问题、无法判断。
   如果部分满足但仍有明确缺失，填写“存在问题”,请严格返回字符串。

2. evidence 必须是一个字符串，不得使用列表。
   应直接引用提供的合同原文。

3. issue 必须是字符串。
   如果符合要求，填写“无”。

4. suggestion 必须是字符串。
   如果符合要求，填写“无需修改”。

5. 不得遗漏任何必填字段。
"""

In [36]:
good_result = structured_llm.invoke(good_prompt)

print(good_result.model_dump())

ValidationError: 1 validation error for ConstractReviewResult
rule_id
  Field required [type=missing, input_value={'status': '符合要求'...estion': '无需修改'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [42]:
debug_llm = llm.with_structured_output(ConstractReviewResult,
                   
                                       include_raw=True)
debug_result = debug_llm.invoke(good_prompt)


In [43]:
print("解析结果：", debug_result["parsed"])
print("解析错误：", debug_result["parsing_error"])
print("原始工具调用：", debug_result["raw"].tool_calls)

解析结果： rule_id='WQ-001' status='符合要求' evidence='6.3 水质超标处理费：甲方交付的污水超过接收标准，但超标倍数不超过2倍，且经乙方评估确认可以处理的，经乙方书面同意接收后，按照原污水处理单价的1.5倍收费。当任一指标超标超过2倍，或可能影响污水处理系统安全稳定运行时，乙方有权拒绝接收。如双方拟继续接收该类严重超标污水，须在接收前另行书面协商处理价格。未能达成一致的，乙方有权拒收。' issue='无' suggestion='无需修改'
解析错误： None
原始工具调用： []
